================================================================================
# ANEXO 3                                
USANDO df_pp_ventas6k y df_producto6k

## PROBABLES MEJORAS A FUTURO        version 35 / 5-nov 03:11


### *Trabajo en curso*

1 Crear lista categorias = df_ventas_final["categoria"].unique()  

2 Crear tabla df_producto con id_producto, producto, id_categoria, categoria

Se observó que:
1. Las ventas contienen productos con la misma denominación, sin ninguna id como para identificarlos. Pertenecen a distintas ventas individuales y tienen distinto precio.

2. Por tal razón es dificil hacer una evaluación seria del resultado de la campaña publicitaria, porque corresponde a distintos productos agrupados, con precio sin adjudicar.

3. La campaña de marketing posee el costo publicitario por producto (generico/agregado) pero no hay como relacionarlo con el costo_unitario, ni con el precio_venta

4. Sacar un precio_promedio, no estoy seguro que sirva mucho, porque la venta podria haberse hecho a un valor menor que el precio_unitario_promedio, debido a la gran dispersión de precios

Hemos visto que para el producto "Adorno de pared", los precios estan distribuidos en 3 segmentos o rangos, la idea es hacer un promedio por rango y
luego renombrar los productos segun el rango, por ej:
"Adorno de pared" => "Adorno de pared R1"

De esta manera en lugar de tener 100 productos genericos los tendriamos divididos por rango de precios y con un nuevo nombre_producto y un nuevo id_producto que los represente
  
.  




ESTE ES EL ENFOQUE QUE VOY A TRATAR DE DESARROLLAR

##S-1  df_producto
Contiene id_producto Unico para un "producto" clasificado por rangos: R1,
R2 y R3 porque los precios estan diferenciados  

Existe 2 denominaciones del producto:  
a) una generica, ej: "Adorno de pared"   
b) otra unica => "Adorno de pared-R1" con  su id asociado: 101  

Este DF contiene ademas:  
'rango': 'R1', 'R2', 'R3', respectivamente  
'media_rango': el valor promedio de ese producto en ese rango de precios  
'precio_rango': el rango de precio unitario para el producto en ese rango



##S-2
Los productos en df_ventas se vinculan por ID Unico, presente en ambas tablas  
Es opcional el nombre unico y estoy evaluando si lo pongo o no

Se agrego ["id_cat"] numerica para facilitar las operaciones y se mantiene["categoria"] para ser convertida como "category" de ser necesaria o borrada

Tambien se reordenaron los campos de los DF

.  

## Por este motivo el DF df_producto, y df_pp_ventas van a ir evolucionando....  
.

In [ ]:
import pandas as pd

# Import dataset ventas final
url = "https://raw.githubusercontent.com/alex-degarate/TT-2C2025-Data-Analitycs-Notebooks/main/PreEntrega/Datasets/"

#df_ventas_final = pd.read_csv( url + "ventas_final.csv")


In [ ]:
# Import dataset clientes
#df_clientes_final = pd.read_csv( url + "clientes_final.csv")

In [ ]:
# Import dataset marketing
df_marketing_final = pd.read_csv( url + "marketing_final.csv")

In [ ]:
df_marketing_final.info()


## a partir de ahora vamos a trabajar con df_pp_ventas2 / 4 (desde disco)

In [5]:
import pandas as pd

In [6]:
# habilito mostrar hasta 100 filas por DF
pd.set_option('display.max_rows', 100)

### archivos para probar puntos finales

In [8]:
# a partir de ahora vamos a trabajar con df_pp_ventas2 ver 6k
url2 ="https://raw.githubusercontent.com/alex-degarate/DAnalytics/main/preproyecto/Datasets/tempo/"
df_pp_ventas = pd.read_csv( "df_pp_ventas6k.csv",index_col=0 )


In [9]:
# @title
df_pp_ventas.head()

,id_venta,id_producto,prod_gener,precio_unit,cantidad,fecha_venta,categoria,rango,precio_rango
0,792,192,Cuadro decorativo,69.94,5,2024-01-02,Decoración,R2,61-86
1,811,293,Lámpara de mesa,105.10,5,2024-01-02,Decoración,R3,83-126
2,1156,353,Secadora,97.96,3,2024-01-02,Electrodomésticos,R3,93-125
3,1372,243,Heladera,114.35,8,2024-01-02,Electrodomésticos,R3,91-123
4,1546,353,Secadora,106.21,4,2024-01-02,Electrodomésticos,R3,93-125


In [10]:
# @title
#df_pp_ventas['producto'] = df_pp_ventas['producto'].replace('Adorno de pared-R1', 'Adorno de pared')
#df_pp_ventas['producto'] = df_pp_ventas['producto'].replace('Adorno de pared-R2', 'Adorno de pared')
#df_pp_ventas['producto'] = df_pp_ventas['producto'].replace('Adorno de pared-R3', 'Adorno de pared')

# Display the updated DataFrame to confirm the changes
#df_pp_ventas[df_pp_ventas["producto"] == 'Adorno de pared'].head(100)

# agrego campo / columna ['precio_rango']
# df_pp_ventas['precio_rango'] = ""


In [ ]:
# @title
## Hacemos una copia para hacer pruebas o si la IA lo arruina
df_pp_ventas2 = df_pp_ventas.copy()
df_pp_ventas2.info()

In [12]:
df_pp_ventas2.head(5)

,id_venta,id_producto,prod_gener,precio_unit,cantidad,fecha_venta,categoria,rango,precio_rango
0,792,192,Cuadro decorativo,69.94,5,2024-01-02,Decoración,R2,61-86
1,811,293,Lámpara de mesa,105.10,5,2024-01-02,Decoración,R3,83-126
2,1156,353,Secadora,97.96,3,2024-01-02,Electrodomésticos,R3,93-125
3,1372,243,Heladera,114.35,8,2024-01-02,Electrodomésticos,R3,91-123
4,1546,353,Secadora,106.21,4,2024-01-02,Electrodomésticos,R3,93-125


## Lista alfabetica de productos

In [52]:
# @title

lista_productos = ["Adorno de pared","Alfombra","Aspiradora","Auriculares","Batidora","Cafetera","Candelabro",
"Consola de videojuegos","Cortinas","Cuadro decorativo","Cámara digital","Elementos de cerámica","Espejo decorativo","Freidora eléctrica","Heladera",
"Horno eléctrico","Jarrón decorativo","Laptop","Lavadora","Lámpara de mesa","Microondas","Parlantes Bluetooth","Plancha de vapor","Proyector","Rincón de plantas","Secadora","SmartWatch","Smartphone","Tablet","Televisor"]

# Creamos una lista de listas, agregando el segundo elemento vacío, para llevar
# control de lo procesado

# agregue un 3er campo   count de cada producto
#                          |
aLista_prod = [[nombre, 0] for nombre in lista_productos]


.  

===== AGREGO EL CONTADOR PARA CADA PRODUCTO, a df_aLista_prod

In [126]:

# Convert the list of lists to a DataFrame
df_aLista_prod = pd.DataFrame(aLista_prod, columns=['Producto', 'Procesado'])

# obtengo el contador para cada producto en df_pp_ventas2
columna3 = df_pp_ventas2["producto"].value_counts().sort_index()

df_columna3 = columna3.reset_index()
# elimino la columna producto, me quedo solo con ["count"]
df_columna3 = df_columna3.drop(columns=['producto'])

#display(df_columna3)
# agrego al df_aLista_prod una nueva columna/Serie ["Count"]
df_aLista_prod['Count'] = df_columna3['count']

#display(df_aLista_prod)

# ESTE DF deberia cargarlo como archivo externo y eliminar los pasos anteriores

,Producto,Procesado,Count
0,Adorno de pared,0,100
1,Alfombra,0,100
2,Aspiradora,0,100
3,Auriculares,0,143
4,Batidora,0,100
5,Cafetera,0,117
6,Candelabro,0,24
7,Consola de videojuegos,0,99
8,Cortinas,0,100
9,Cuadro decorativo,0,100


# Utilizaremos el que esta en curso df_producto5
Para hacer de enlace como clave primaria entre ventas y marketing


## El DF df_producto5 es el que esta actualizado y el que debe cargarse desde github

In [16]:
url2 ="https://raw.githubusercontent.com/alex-degarate/DAnalytics/main/preproyecto/Datasets/tempo/"

# OJO ! se levanta como df_producto usar ver 6k !!
# df_producto = pd.read_csv( url2 + "df_producto6k.csv", index_col=0 )
df_producto = pd.read_csv( "df_producto6k.csv", index_col=0 )
df_producto.info()



<class 'pandas.core.frame.DataFrame'>
Index: 90 entries, 0 to 89
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_producto   90 non-null     int64  
 1   producto      90 non-null     object 
 2   id_cat        90 non-null     int64  
 3   categoria     90 non-null     object 
 4   rango         90 non-null     object 
 5   media_rango   90 non-null     float64
 6   precio_rango  90 non-null     object 
 7   prod_gener    90 non-null     object 
dtypes: float64(1), int64(2), object(5)
memory usage: 6.3+ KB


In [24]:
# @title
# mostrar solo las columnas indicadas

#display(df_producto[['id_producto', 'producto', 'rango','media_rango', 'precio_rango']].head(90))

# reordenamos las columnas UNA SOLA VEZ
#df_producto = df_producto[['id_producto','producto','prod_gener','rango','id_cat','categoria',
#'media_rango','precio_rango']]

display(df_producto.head())



,id_producto,producto,prod_gener,rango,id_cat,categoria,media_rango,precio_rango
0,101,Adorno de pared-R1,Adorno de pared,R1,1,Decoración,44.39,25-66
1,102,Adorno de pared-R2,Adorno de pared,R2,1,Decoración,78.29,66-91
2,103,Adorno de pared-R3,Adorno de pared,R3,1,Decoración,105.74,91-120
3,111,Alfombra-R1,Alfombra,R1,1,Decoración,41.55,28-54
4,112,Alfombra-R2,Alfombra,R2,1,Decoración,71.00,54-89


In [26]:
df_pp_ventas2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2998 entries, 0 to 2997
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_venta      2998 non-null   int64  
 1   id_producto   2998 non-null   int64  
 2   prod_gener    2998 non-null   object 
 3   rango         2998 non-null   object 
 4   precio_unit   2998 non-null   float64
 5   cantidad      2998 non-null   int64  
 6   fecha_venta   2998 non-null   object 
 7   id_cat        2998 non-null   int64  
 8   categoria     2998 non-null   object 
 9   precio_rango  2998 non-null   object 
dtypes: float64(1), int64(4), object(5)
memory usage: 234.3+ KB


### juntamos id_cat de producto en df_pp_ventas2

In [ ]:
# Merge df_pp_ventas2 with df_producto on the 'id_producto' column
# We only need the 'id_producto' and 'id_cat' columns from df_producto for the merge
#df_pp_ventas2 = pd.merge(df_pp_ventas2, df_producto[['id_producto', 'id_cat']], on='id_producto', how='left')

# Display the first few rows of the updated df_pp_ventas2 to see the new 'id_cat' column
#display(df_pp_ventas2.head())

# reordenamos las columnas
#df_pp_ventas2 = df_pp_ventas2[['id_venta','id_producto','prod_gener','rango','precio_unit',
#'cantidad','fecha_venta','id_cat','categoria','precio_rango']]

# Display the info of the updated df_pp_ventas2 to confirm the new column and its dtype
df_pp_ventas2.info()


In [ ]:
# Display the first few rows of the updated df_pp_ventas2
# display(df_pp_ventas2.head())

In [ ]:
'''
df_pp_ventas2["categoria"] = df_pp_ventas2["categoria"].astype('category')

# You can check the data type after conversion
print(df_pp_ventas2["categoria"].dtype)
'''

### Aparecieron imprevistos que complican el codigo, y tengo que reintroducir el nombre generico del producto, independiente del rango a que pertenezca.   

### Aparecieron datos faltantes y NO SE POR QUE !!!  

### Voy a reiniciar el codigo desde Adorno de pared en adelante...

## REVISAR la info que no haya datos vacios / nan etc

### Tampoco debo de olvidarme de actualizar y guardar df_producto y df_pp_ventas
## **Deben ir juntos y sincronizados !!!**


================================================================================

.

# PASOS PARA AUTOMATIZAR
## PARA TODOS LOS PRODUCTOS

### 1. Recorrer La lista de productos,
&emsp; si el 2do campo esta en cero significa que no fue procesado, 1=completo
lista_productos = [["Adorno de pared", 1]

### 2. OBTENER NOMBRE DEL PRODUCTO A PROCESAR

### 3. CREAR EL DF TEMPORAL

### 4. VISUALIZAR ESTADISTICA CON DESCRIBE

### 5 CALCULAR RANGOS, MEDIA

### 6. CALCULAR Cuartiles (0.33 y 0.66)

### 7. GRAFICAR HISTOGRAMA

### 8. REEMPLAZAR DATOS EN DF TEMPORAL

### 9. ACTUALIZAR df_pp_ventas


## 2. OBTENER NOMBRE DEL PRODUCTO A PROCESAR

In [847]:

nLenght = df_aLista_prod.shape[0]
'''
df_aLista_prod.iat[ 0, 1] = 1  # Adorno de pared
df_aLista_prod.iat[ 1, 1] = 1  # Alfombra
df_aLista_prod.iat[ 2, 1] = 1  # Aspiradora
df_aLista_prod.iat[ 3, 1] = 1  # Auriculares
df_aLista_prod.iat[ 4, 1] = 1  # Batidora
df_aLista_prod.iat[ 5, 1] = 1  # Cafetera
df_aLista_prod.iat[ 6, 1] = 1  # Candelabro
df_aLista_prod.iat[ 7, 1] = 1  # Consola de videojuegos
df_aLista_prod.iat[ 8, 1] = 1  # Cortinas
df_aLista_prod.iat[ 9, 1] = 1  # Cuadro decorativo
df_aLista_prod.iat[10, 1] = 1  # Cámara digital
df_aLista_prod.iat[11, 1] = 1  # Elementos de cerámica
df_aLista_prod.iat[12, 1] = 1  # Espejo decorativo
df_aLista_prod.iat[13, 1] = 1  # Freidora eléctrica
df_aLista_prod.iat[14, 1] = 1  # Heladera
df_aLista_prod.iat[15, 1] = 1  # Horno eléctrico
df_aLista_prod.iat[16, 1] = 1  # Jarrón decorativo
df_aLista_prod.iat[17, 1] = 1  # Laptop
df_aLista_prod.iat[18, 1] = 1  # Lavadora
df_aLista_prod.iat[19, 1] = 1  # Lámpara de mesa
df_aLista_prod.iat[20, 1] = 1  # Microondas
df_aLista_prod.iat[21, 1] = 1  # Parlantes Bluetooth
df_aLista_prod.iat[22, 1] = 1  # Plancha de vapor
df_aLista_prod.iat[23, 1] = 1  # Proyector
df_aLista_prod.iat[24, 1] = 1  # Rincón de plantas
df_aLista_prod.iat[25, 1] = 1  # Secadora
df_aLista_prod.iat[26, 1] = 1  # SmartWatch
df_aLista_prod.iat[27, 1] = 1  # Smartphone
df_aLista_prod.iat[28, 1] = 1  # Tablet
#df_aLista_prod.iat[29, 1] = 1  # Televisor
'''
for i in range( nLenght):

    if df_aLista_prod.iat[i, 1] == 0:
       prod_actual = df_aLista_prod.iat[i, 0]      # nombre producto
       break

prod_actual = prod_actual
num_elem = df_aLista_prod.iat[i, 2]
print(f"El primer item a procesar es {[i]}: {prod_actual}, tiene {num_elem} registros \n")


El primer item a procesar es [29]: Televisor, tiene 100 registros 



In [820]:
display(df_aLista_prod)

,Producto,Procesado,Count
0,Adorno de pared,1,100
1,Alfombra,1,100
2,Aspiradora,1,100
3,Auriculares,1,143
4,Batidora,1,100
5,Cafetera,1,117
6,Candelabro,1,24
7,Consola de videojuegos,1,99
8,Cortinas,1,100
9,Cuadro decorativo,1,100


# Producto ACTUAL

In [865]:
print( f"df_{prod_actual} \n")
print( df_pp_ventas2[ df_pp_ventas2["prod_gener"] == prod_actual].count())

df_Televisor 

id_venta        100
id_producto     100
prod_gener      100
precio_unit     100
cantidad        100
fecha_venta     100
categoria       100
rango           100
precio_rango    100
dtype: int64


In [866]:
# df_pp_ventas2 = df_pp_ventas.sort_values(by="producto", ascending=True)
df_pp_ventas2[df_pp_ventas2["prod_gener"] == prod_actual].head()


,id_venta,id_producto,prod_gener,precio_unit,cantidad,fecha_venta,categoria,rango,precio_rango
99,2822,392,Televisor,57.34,1,2024-01-13,Electrónica,R2,57-85
139,2402,393,Televisor,117.53,9,2024-01-17,Electrónica,R3,85-124
150,2792,392,Televisor,84.96,5,2024-01-18,Electrónica,R2,57-85
201,2982,391,Televisor,26.70,9,2024-01-24,Electrónica,R1,26-57
209,2502,392,Televisor,77.83,8,2024-01-25,Electrónica,R2,57-85


## 3. CREAR EL DF TEMPORAL
### df_actual corresponde a df_xxxx.....
la idea es generalizarlo para el procesamiento de los otros productos

In [867]:
# Generamos una tabla aislada para el producto actual => xxxx

# df_actual corresponde a
df_actual = df_pp_ventas2[df_pp_ventas2["prod_gener"] == prod_actual].copy()

media = df_actual["precio_unit"].mean()
print(f"El precio promedio para '{prod_actual}' es: {media:.2f} \n")

df_actual.describe()


El precio promedio para 'Televisor' es: 73.82 



,id_venta,id_producto,precio_unit,cantidad
count,100.000000,100.000000,100.000000,100.000000
mean,2499.360000,392.000000,73.815300,6.320000
std,293.154095,0.828775,27.476884,3.281044
min,2002.000000,391.000000,26.700000,1.000000
25%,2249.500000,391.000000,51.960000,3.750000
50%,2497.000000,392.000000,69.930000,7.000000
75%,2747.000000,393.000000,97.477500,9.000000
max,2991.000000,393.000000,123.480000,12.000000


In [868]:
df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 99 to 2956
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_venta      100 non-null    int64  
 1   id_producto   100 non-null    int64  
 2   prod_gener    100 non-null    object 
 3   precio_unit   100 non-null    float64
 4   cantidad      100 non-null    int64  
 5   fecha_venta   100 non-null    object 
 6   categoria     100 non-null    object 
 7   rango         100 non-null    object 
 8   precio_rango  100 non-null    object 
dtypes: float64(1), int64(3), object(5)
memory usage: 7.8+ KB


# OJO
### Completar en df_actual la columna ["rango"]

### 4. VISUALIZAR ESTADISTICA CON DESCRIBE

 Obtenemos info relevante mediante describe()

### 5 CALCULAR RANGOS, MEDIA

In [825]:


'''
The .describe() method in pandas returns a new DataFrame containing the
descriptive statistics.
You can access the individual statistics (like min, max, mean, etc.) from this
DataFrame using standard indexing.
'''
# Obtenemos la salida describe del DataFrame
#-------------------------------------------
def obtener_descripcion_DF(df):
    return df.describe()
#df_desc = obtener_descripcion_DF( df_actual["precio_unit"])

df_desc = df_actual["precio_unit"].describe()

# Accedemos a valores especificos
min_precio = int( round( df_desc['min'])) -1
max_precio = int( round( df_desc['max'])) + 1
mean_precio = df_desc['mean']

print(f"{prod_actual} \n")
print(f"PUnit Minimo: {min_precio}")
print(f"PUnit Maximo: {max_precio}")
print(f"PUnit Medio : {mean_precio:.2f} \n")

display (df_desc)

Televisor 

PUnit Minimo: 26
PUnit Maximo: 124
PUnit Medio : 73.82 



,precio_unit
count,100.000000
mean,73.815300
std,27.476884
min,26.700000
25%,51.960000
50%,69.930000
75%,97.477500
max,123.480000


In [857]:
def mostrar_dispersion():
    if (max_precio / min_precio) > 2:
        print(f'En {prod_actual} => HAY DISPERSION de precios !!!')
    else:
        print(f'En {prod_actual} No hay dispersion de precios')



### 6. CALCULAR Cuartiles (0.33 y 0.66)

In [856]:
def calcular_cuartiles():
    # Calculamos percentilo 33 de 'precio_unit'
    q33_data = df_actual["precio_unit"].quantile(0.33)

    # Calculamos el percentilo 66 de 'precio_unit'
    q66_data = df_actual["precio_unit"].quantile(0.66)

    print(f"El percentilo 33 del precio_unit para '{prod_actual}' es: {q33_data:.2f}")
    print(f"El percentilo 66 del precio_unit para '{prod_actual}' es: {q66_data:.2f}")


### 7. GRAFICAR HISTOGRAMA

In [855]:
def graficar_histograma():
    # Agregamos un plot para visualizar la distribución
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np

    sns.histplot( df_actual["precio_unit"], bins=100, kde=True, edgecolor="black")
    plt.title("Distribución de precio_unit - "+ prod_actual)
    plt.xlabel("Precio Unitario ($)")
    plt.ylabel("Frecuencia")

    # Seteamos x-axis ticks a intervalo s de 10
    plt.xticks(np.arange(0, df_actual ["precio_unit"].max() + 10, 10))
    plt.show()



###  CALCULAR RANGOS
Aca calculamos los puntos de corte y los asignamos a df_actual["precio_rango"]

In [869]:
# Definimos los bins (puntos de corte)
r1_linf = min_precio
r1_lsup = int( round( q33_data))
r2_linf = r1_lsup
r2_lsup = int( round( q66_data))
r3_linf = r2_lsup
r3_lsup = max_precio

#print(f"{r2_lsup} \n")

#bins = [26, 61, 87, 130]
bins = [r1_linf, r1_lsup, r3_linf, r3_lsup]
print( f"{prod_actual}  {bins}")
print("\n")

# Definimos las etiquetas para los rangos
# labels = ['26-61', '61-86', '86-125']
rg1 = f"{r1_linf}-{r1_lsup}"
rg2 = f"{r2_linf}-{r2_lsup}"
rg3 = f"{r3_linf}-{r3_lsup}"
labels = [ rg1, rg2, rg3 ]


# Creamos una nueva columna 'precio_rango' para estratificar 'precio_unit' dentro de los cut-off
df_actual['precio_rango'] = pd.cut( df_actual['precio_unit'], bins=bins, labels=labels, right=False)

# Mostramos la cantidad de items precios en cada rango de precios
display( df_actual['precio_rango'].value_counts())

# Display the first few rows with the new column
print("\n===================== DF_ACTUAL =====================\n")
display( df_actual.head(10))


Televisor  [26, 57, 85, 124]




,count
precio_rango,
26-57,34
85-124,34
57-85,32



===================== DF_ACTUAL =====================



,id_venta,id_producto,prod_gener,precio_unit,cantidad,fecha_venta,categoria,rango,precio_rango
99,2822,392,Televisor,57.34,1,2024-01-13,Electrónica,R2,57-85
139,2402,393,Televisor,117.53,9,2024-01-17,Electrónica,R3,85-124
150,2792,392,Televisor,84.96,5,2024-01-18,Electrónica,R2,57-85
201,2982,391,Televisor,26.70,9,2024-01-24,Electrónica,R1,26-57
209,2502,392,Televisor,77.83,8,2024-01-25,Electrónica,R2,57-85
218,2252,393,Televisor,89.27,8,2024-01-26,Electrónica,R3,85-124
228,2222,391,Televisor,33.19,9,2024-01-27,Electrónica,R1,26-57
252,2522,393,Televisor,94.98,2,2024-01-30,Electrónica,R3,85-124
302,2832,391,Televisor,49.23,4,2024-02-04,Electrónica,R1,26-57
363,2952,393,Televisor,112.58,5,2024-02-11,Electrónica,R3,85-124


.

## CALCULAMOS LA MEDIA PARA CADA RANGO: R1, R2, R3 DEL PRODUCTO

In [829]:
# Yo RENOMBRE la desc producto, pero tal vez seria mejor agregar un
# campo con la clasif x RANGO => LO ESTOY RECONSIDERANDO

# Observamos que hay 3 rangos de datos para xxxxx
# Calculamos esos rangos aproxim dividiendolos en terciles

# estos limites son enteros

# Cambie este array para usarlo en una funcion
# limite   suma cant  Med
rango1 = [  0,   0,  0.0 ]
rango2 = [  0,   0,  0.0 ]
rango3 = [  0,   0,  0.0 ]

# Recorre df_actual  la tabla sacada de df_pp_ventas, para producto "xxxx"

nLenght = df_actual.shape[0]
print(f"{prod_actual} = {nLenght}")

#        df_cuadro_decorativo.iat[ num_fila, num_col ]  con indice= 0
# print( df_cuadro_decorativo.iat[0, 2])

# Recorremos la tabla viendo el PU en que rango cae y lo clasificamos
# el valor lo guardamos en rango y lo agregamos al campo precio_rango

for i in range( nLenght):

    pu = df_actual.iat[i, 3]      # Precio Unitario

    nomP = df_actual.iat[i, 2]    # nombre producto

    if pu > r1_linf and pu <= r1_lsup:
        rango1[0] += pu        # suma pu
        rango1[1] += 1         # count
        #df_actual.iat[i, 2] = nomP #+ "-R1"
        df_actual.iat[i, 7] = "R1"    # actualizo rango del producto

    elif pu > r2_linf and pu <= r2_lsup:
        rango2[0] += pu        # suma pu
        rango2[1] += 1         # count
        #df_actual.iat[i, 2] = nomP #+ "-R2"
        df_actual.iat[i, 7] = "R2"

    elif pu > r3_linf and pu <= r3_lsup:
        rango3[0] += pu        # suma pu
        rango3[1] += 1         # count
        #df_actual.iat[i, 2] = nomP #+ "-R3"
        df_actual.iat[i, 7] = "R3"


if df_actual.iat[i, 0] == 0:
   display( f"{i} {df_actual.iat[i, 1]} ERROR \n")


Televisor = 100


## Las medias_rango y precio_rango se deben copiar en [df_producto]

In [830]:
# Calculamos las medias x rango
# rango1[0] = suma pu, rango1[1] = contador
media_rango1 = round( rango1[0]/rango1[1], 2)
media_rango2 = round( rango2[0]/rango2[1], 2)
media_rango3 = round( rango3[0]/rango3[1], 2)

#Visualizamos las medias
print(f"{prod_actual}")
print(f"media_rango1 : {media_rango1}")
print(f"media_rango2 : {media_rango2}")
print(f"media_rango3 : {media_rango3}")

# En las celdas de mas abajo lo graficamos

Televisor
media_rango1 : 43.92
media_rango2 : 71.15
media_rango3 : 106.22


In [870]:
# Copiar desde df_producto id_producto a df_pp_ventas para articulo xxxx
tabla_codigos = df_producto[df_producto["producto"].str.contains( prod_actual, na=False)][["id_producto", "producto", "rango", "prod_gener"]].copy()

display(tabla_codigos)


,id_producto,producto,rango,prod_gener
87,391,Televisor-R1,R1,Televisor
88,392,Televisor-R2,R2,Televisor
89,393,Televisor-R3,R3,Televisor


## En DF_PRODUCTO Guardamos las media_rango y precio_rango

In [832]:
# Obtiene el index de la primera fila en tabla_codigos que corresponde a los
# registos del df_producto para actualizar media_rango y media_precio
recno1 = tabla_codigos.index[0]
recno2 = tabla_codigos.index[1]
recno3 = tabla_codigos.index[2]
print(f"{prod_actual}")
print( recno1, recno2, recno3)

#print(f"el index de la primera fila para + '{prod_actual}' en df_producto es: {recno1}"

# Guardamos las media_rango
df_producto.iat[recno1, 5] = media_rango1
df_producto.iat[recno2, 5] = media_rango2
df_producto.iat[recno3, 5] = media_rango3

# Guardamos las precio_rango
df_producto.iat[recno1, 6] = rg1
df_producto.iat[recno2, 6] = rg2
df_producto.iat[recno3, 6] = rg3
#print( df_producto.iat[recno3, 6])

# If you want all indices, you can simply access the index attribute
all_indices = tabla_codigos.index
#print(f"All indices for '{prod_actual}' in df_producto are: {list(all_indices)}")

Televisor
87 88 89


In [871]:
display(df_producto.loc[all_indices])

,id_producto,producto,id_categoria,categoria,rango,media_rango,precio_rango,prod_gener
87,391,Televisor-R1,3,Electrónica,R1,43.92,26-57,Televisor
88,392,Televisor-R2,3,Electrónica,R2,71.15,57-85,Televisor
89,393,Televisor-R3,3,Electrónica,R3,106.22,85-124,Televisor


##

### Recorro df_actual completando los datos:  [id_producto]

In [834]:

# OJO ACA si esta habilitado el unnamed index !!! no va a coincidir
articulo = tabla_codigos.iat[0, 3] # nombre producto
cod1 = tabla_codigos.iat[0, 0] # id_producto
cod2 = tabla_codigos.iat[1, 0]
cod3 = tabla_codigos.iat[2, 0]

nLenght2 = len( df_actual )
#print(f"{nLenght2}")
print(f"producto: {articulo}")
print(f"code0 {cod1}")
print(f"code1 {cod2}")
print(f"code2 {cod3}")
#print(f"code1 {tabla_codigos.iat[0, 1]}")  # nombre producto
#print(f"code2 {tabla_codigos.iat[0, 2]}")  # rango R1, R2,R3
#print(f"code3 {tabla_codigos.iat[0, 3]}")  # prod_gener

for i in range( nLenght2):

    if df_actual.iat[i, 2] == articulo:  # nombre producto

       if df_actual.iat[i, 7] == "R1":
          df_actual.iat[i, 1] = cod1

       elif df_actual.iat[i, 7] == "R2":
          df_actual.iat[i, 1] = cod2

       elif df_actual.iat[i, 7] == "R3":
          df_actual.iat[i, 1] = cod3


producto: Televisor
code0 391
code1 392
code2 393


In [872]:
df_actual[ df_actual["prod_gener"] == prod_actual].head(10)


,id_venta,id_producto,prod_gener,precio_unit,cantidad,fecha_venta,categoria,rango,precio_rango
99,2822,392,Televisor,57.34,1,2024-01-13,Electrónica,R2,57-85
139,2402,393,Televisor,117.53,9,2024-01-17,Electrónica,R3,85-124
150,2792,392,Televisor,84.96,5,2024-01-18,Electrónica,R2,57-85
201,2982,391,Televisor,26.70,9,2024-01-24,Electrónica,R1,26-57
209,2502,392,Televisor,77.83,8,2024-01-25,Electrónica,R2,57-85
218,2252,393,Televisor,89.27,8,2024-01-26,Electrónica,R3,85-124
228,2222,391,Televisor,33.19,9,2024-01-27,Electrónica,R1,26-57
252,2522,393,Televisor,94.98,2,2024-01-30,Electrónica,R3,85-124
302,2832,391,Televisor,49.23,4,2024-02-04,Electrónica,R1,26-57
363,2952,393,Televisor,112.58,5,2024-02-11,Electrónica,R3,85-124


.

# ACTUALIZAMOS df_pp_ventas

In [836]:
# Tomar con pinzas ..
# Update df_pp_ventas with the modified rows from df_actual

print(f"Actualizando pp_ventas2 con 'df_{prod_actual}")

# df_ventas2 NO CONTIENE precio_rango ??? => que hago ?
df_pp_ventas2.update( df_actual[['id_producto', 'rango', 'precio_rango']] )



Actualizando pp_ventas2 con 'df_Televisor


In [873]:

# Display the updated df_pp_ventas

#df_pp_ventas2[df_pp_ventas2['prod_gener'].str.contains( prod_actual)].head(100)
df_pp_ventas2[df_pp_ventas2['prod_gener'] == prod_actual].head()

,id_venta,id_producto,prod_gener,precio_unit,cantidad,fecha_venta,categoria,rango,precio_rango
99,2822,392,Televisor,57.34,1,2024-01-13,Electrónica,R2,57-85
139,2402,393,Televisor,117.53,9,2024-01-17,Electrónica,R3,85-124
150,2792,392,Televisor,84.96,5,2024-01-18,Electrónica,R2,57-85
201,2982,391,Televisor,26.70,9,2024-01-24,Electrónica,R1,26-57
209,2502,392,Televisor,77.83,8,2024-01-25,Electrónica,R2,57-85


In [874]:
df_actual[df_actual["id_producto"] == 0]

,id_venta,id_producto,prod_gener,precio_unit,cantidad,fecha_venta,categoria,rango,precio_rango


In [875]:
df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 99 to 2956
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   id_venta      100 non-null    int64   
 1   id_producto   100 non-null    int64   
 2   prod_gener    100 non-null    object  
 3   precio_unit   100 non-null    float64 
 4   cantidad      100 non-null    int64   
 5   fecha_venta   100 non-null    object  
 6   categoria     100 non-null    object  
 7   rango         100 non-null    object  
 8   precio_rango  100 non-null    category
dtypes: category(1), float64(1), int64(3), object(4)
memory usage: 7.3+ KB


# ====================================================

## Guardamos los dataframe en uso...

In [29]:
# Guardamos df_pp_ventas2[] junto con el indice
df_pp_ventas2.to_csv('df_pp_ventas6k2.csv', index=True)

In [30]:
# Guardamos df_producto[] junto con el indice
df_producto.to_csv('df_producto6k2.csv', index=True)

In [ ]:
# @title
# Guardamos df_actual
#index_col=0
#sfile = 'df_' + prod_actual + '.csv'
#print(f" guardando.. {sfile}")
#df_actual.to_csv(sfile, index=True)


In [793]:
# Guardamos df_aLista_prod
df_producto.to_csv('df_aLista_prod.csv', index=True)

.  

===============================================================================

A PARTIR DE ACA EL REPROCESAMIENTO DE VENTAS Y MARKETING


In [ ]:
# @title
'''
# Calculate the sum and count for each range using boolean indexing

rango1_data = df_adorno_pared[df_adorno_pared['precio_range'] == '25-66']
rango1_sum = rango1_data['precio_unit'].sum()
rango1_count = rango1_data.shape[0]

rango2_data = df_adorno_pared[df_adorno_pared['precio_range'] == '63-93']
rango2_sum = rango2_data['precio_unit'].sum()
rango2_count = rango2_data.shape[0]

rango3_data = df_adorno_pared[df_adorno_pared['precio_range'] == '93-120']
rango3_sum = rango3_data['precio_unit'].sum()
rango3_count = rango3_data.shape[0]


print(f"Range 25-66: Sum = {rango1_sum:.2f}, Count = {rango1_count}, Average = {(rango1_sum / rango1_count):.2f}")
print(f"Range 63-93: Sum = {rango2_sum:.2f}, Count = {rango2_count}, Average = {(rango2_sum / rango2_count):.2f}")
print(f"Range 93-120: Sum = {rango3_sum:.2f}, Count = {rango3_count}, Average = {(rango3_sum / rango3_count):.2f}")
'''


'\n# Calculate the sum and count for each range using boolean indexing\n\nrango1_data = df_adorno_pared[df_adorno_pared[\'precio_range\'] == \'25-66\']\nrango1_sum = rango1_data[\'precio_unit\'].sum()\nrango1_count = rango1_data.shape[0]\n\nrango2_data = df_adorno_pared[df_adorno_pared[\'precio_range\'] == \'63-93\']\nrango2_sum = rango2_data[\'precio_unit\'].sum()\nrango2_count = rango2_data.shape[0]\n\nrango3_data = df_adorno_pared[df_adorno_pared[\'precio_range\'] == \'93-120\']\nrango3_sum = rango3_data[\'precio_unit\'].sum()\nrango3_count = rango3_data.shape[0]\n\n\nprint(f"Range 25-66: Sum = {rango1_sum:.2f}, Count = {rango1_count}, Average = {(rango1_sum / rango1_count):.2f}")\nprint(f"Range 63-93: Sum = {rango2_sum:.2f}, Count = {rango2_count}, Average = {(rango2_sum / rango2_count):.2f}")\nprint(f"Range 93-120: Sum = {rango3_sum:.2f}, Count = {rango3_count}, Average = {(rango3_sum / rango3_count):.2f}")\n'

In [ ]:
# @title
'''
import seaborn as sns
import matplotlib.pyplot as plt

# Create a KDE plot
sns.kdeplot(df_adorno_pared["precio_unit"], fill=True)
plt.title("Kernel Density Estimate of Precio Unitario for Adorno de pared")
plt.xlabel("Precio Unitario ($)")
plt.ylabel("Density")
plt.show()
'''

'\nimport seaborn as sns\nimport matplotlib.pyplot as plt\n\n# Create a KDE plot\nsns.kdeplot(df_adorno_pared["precio_unit"], fill=True)\nplt.title("Kernel Density Estimate of Precio Unitario for Adorno de pared")\nplt.xlabel("Precio Unitario ($)")\nplt.ylabel("Density")\nplt.show()\n'

In [ ]:
#df_marketing.info()
#df_marketing.head()

In [ ]:
#df_marketing2 = df_marketing.sort_values(by="producto", ascending=True)
#df_marketing2.info()
# df_canales = df_marketing.copy()
# df_canales.head()
#df_marketing2.head(3)

In [ ]:
# df_marketing2.to_csv('df_marketing2.csv', index=False)

In [ ]:
#df_ventas_final.to_csv('df_ventas_final.csv', index=False)
# df_producto.to_csv('df_producto.csv', index=False)

In [ ]:
# Agregar a df_ventas_final la columnas "prod_id" y cat_id
df_ventas_final = pd.merge(df_ventas_final, df_producto[['prod_id', 'prod_name', 'cat_id', 'cat_name']], left_on='producto', right_on='prod_name', how='left')

# Drop the redundant 'prod_name' and 'cat_name' columns from the merge
df_ventas_final.drop(['prod_name', 'cat_name'], axis=1, inplace=True)

# Display the updated DataFrame
display(df_ventas_final.head())

In [ ]:
df_ventas_final = df_ventas_final[[ "id_venta",	"prod_id", "producto","precio_unit", "cantidad",
                                   "fecha_venta", "cat_id","categoria", "valor_venta"]]
df_ventas_final.head()

In [ ]:
# @title
# guardamos df_producto para su uso posterior, incrementar el digito al guardar

def guardar_df_a_csv(df, nombre_archivo):
    df.to_csv(nombre_archivo, index=True)
    print(f"DataFrame guardado como {nombre_archivo}")

# Llamada a la función para guardar df_
# guardar_df_a_csv( df_producto4, 'df_producto5.csv')

# df_producto.to_csv('df_producto5.csv', index=True)

# df_pp_ventas.to_csv('df_pp_ventas4.csv', index=True)